# 🔍 Project 01 — Exploratory Data Analysis (EDA)
### Titanic Dataset: Who Survived?

**Author:** Reabetswe Maine  
**University:** North West University  
**Focus:** Data Science & Machine Learning  

---

## 📌 What This Project Covers
- Loading and exploring a real dataset
- Data cleaning (handling missing values)
- Visualising patterns in data
- Building a simple ML model (Logistic Regression)
- Evaluating model performance

> 🧠 This is my **first ML project** as I begin my AI/ML journey!


## 📦 Step 1: Import Libraries

In [ ]:
# Core libraries every ML engineer uses daily
import numpy as np          # numerical computing
import pandas as pd         # data manipulation
import matplotlib.pyplot as plt  # plotting
import seaborn as sns       # beautiful visualizations

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All libraries imported successfully!')

## 📂 Step 2: Load the Dataset

In [ ]:
# Load Titanic dataset directly from seaborn's built-in datasets
df = sns.load_dataset('titanic')

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')
df.head(10)

## 🔎 Step 3: Understand the Data

In [ ]:
# Get basic info about each column
df.info()

In [ ]:
# Statistical summary of numerical columns
df.describe()

In [ ]:
# Check for missing values — very important in real data!
missing = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing,
    'Percentage (%)': missing_percent.round(2)
})

print('🔴 Missing Values per Column:')
missing_df[missing_df['Missing Values'] > 0]

## 📊 Step 4: Exploratory Data Analysis (EDA) — Visualisations

In [ ]:
# Survival count
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Survival count
survival_counts = df['survived'].value_counts()
axes[0].bar(['Did Not Survive', 'Survived'], survival_counts.values, 
            color=['#e74c3c', '#2ecc71'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Survival Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Passengers')
for i, v in enumerate(survival_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Plot 2: Survival by gender
gender_survival = df.groupby(['sex', 'survived']).size().unstack()
gender_survival.plot(kind='bar', ax=axes[1], color=['#e74c3c', '#2ecc71'], 
                     edgecolor='white', rot=0)
axes[1].set_title('Survival by Gender', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Gender')
axes[1].set_ylabel('Count')
axes[1].legend(['Did Not Survive', 'Survived'])

plt.tight_layout()
plt.savefig('survival_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight: Women had a much higher survival rate than men!')

In [ ]:
# Age distribution by survival
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age distribution
df['age'].dropna().hist(bins=30, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Age Distribution of Passengers', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Count')

# Age by survival
df[df['survived'] == 0]['age'].dropna().hist(bins=25, alpha=0.6, ax=axes[1], 
                                               color='#e74c3c', label='Did Not Survive')
df[df['survived'] == 1]['age'].dropna().hist(bins=25, alpha=0.6, ax=axes[1], 
                                               color='#2ecc71', label='Survived')
axes[1].set_title('Age Distribution by Survival', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].legend()

plt.tight_layout()
plt.show()
print('💡 Insight: Younger children had higher survival rates!')

In [ ]:
# Survival by passenger class
fig, ax = plt.subplots(figsize=(8, 5))
pclass_survival = df.groupby('pclass')['survived'].mean() * 100
bars = ax.bar(['1st Class', '2nd Class', '3rd Class'], pclass_survival.values,
               color=['#f1c40f', '#95a5a6', '#cd6155'], edgecolor='white', linewidth=1.5)
ax.set_title('Survival Rate by Passenger Class (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('Survival Rate (%)')
for bar, val in zip(bars, pclass_survival.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
            f'{val:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print('💡 Insight: 1st class passengers had a 63% survival rate vs 24% for 3rd class!')

## 🧹 Step 5: Data Cleaning & Feature Engineering

In [ ]:
# Select features for our model
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
target = 'survived'

# Create a clean copy
df_model = df[features + [target]].copy()

# Fill missing age values with median (common technique)
df_model['age'].fillna(df_model['age'].median(), inplace=True)

# Encode 'sex': male=0, female=1
le = LabelEncoder()
df_model['sex'] = le.fit_transform(df_model['sex'])

# Drop any remaining NaN rows
df_model.dropna(inplace=True)

print(f'✅ Clean dataset: {df_model.shape[0]} rows')
print(f'Missing values: {df_model.isnull().sum().sum()}')
df_model.head()

## 🤖 Step 6: Build & Train ML Model

In [ ]:
# Split into features (X) and target (y)
X = df_model[features]
y = df_model[target]

# Split into training and testing sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

# Train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print('\n✅ Model trained successfully!')

## 📈 Step 7: Evaluate the Model

In [ ]:
# Make predictions
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'🎯 Model Accuracy: {accuracy:.2%}')
print()
print('📊 Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Did Not Survive', 'Survived']))

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Did Not Survive', 'Survived'],
            yticklabels=['Did Not Survive', 'Survived'])
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

## 🏁 Summary & Key Learnings

### What I Built
A complete ML pipeline from raw data to predictions on the Titanic dataset.

### Key Insights Found
- 👩 **Gender mattered most** — women had significantly higher survival rates
- 🎫 **Class mattered** — 1st class passengers survived at 63% vs 24% for 3rd class
- 👶 **Children were prioritized** — younger passengers had higher survival rates

### Skills Practised
- ✅ Data loading and exploration with Pandas
- ✅ Handling missing values
- ✅ Data visualisation with Matplotlib & Seaborn
- ✅ Feature encoding (LabelEncoder)
- ✅ Train/test split
- ✅ Logistic Regression with Scikit-learn
- ✅ Model evaluation (accuracy, confusion matrix)

### Next Steps
- 🔄 Try other models: Random Forest, Decision Tree, SVM
- 🔧 Feature engineering: extract title from name, family size
- 📊 Hyperparameter tuning
coming soon
---
*Project 01 of my ML Portfolio | Reabetswe Maine | North West University*